# Unit 3 Assignment: Production Advanced RAG System

In [1]:
%pip install python-dotenv rank-bm25 sentence-transformers langchain langchain-google-genai numpy --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 2.9 MB/s eta 0:00:00


In [2]:
from dotenv import load_dotenv
import os, getpass

load_dotenv()
if not os.getenv("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter Google API Key: ")
print("Setup complete.")

Enter Google API Key: ··········
Setup complete.


## Part 1 — Document Corpus Setup

In [3]:
corpus = [
    # Attention / Transformers
    "Transformers use self-attention mechanisms to compute weighted sums of value vectors based on query-key similarity.",
    "The attention mechanism allows each token to attend to every other token in the sequence, capturing long-range dependencies.",
    "Multi-head attention runs several attention operations in parallel, each learning different relationship patterns.",

    # Neural Network Training (3 related but distinct)
    "Gradient descent updates model weights by computing the gradient of the loss and stepping in the negative direction.",
    "Backpropagation computes gradients layer by layer using the chain rule, enabling efficient training of deep networks.",
    "The Adam optimizer combines momentum and adaptive learning rates, making it the default choice for training most LLMs.",

    # Other AI/ML topics
    "BERT is a bidirectional encoder pre-trained using masked language modelling on large text corpora.",
    "Retrieval Augmented Generation (RAG) combines a retriever with a language model to produce grounded, factual answers.",
    "Neural networks learn by adjusting weights through backpropagation to minimize a task-specific loss function.",

    # Technical jargon doc (BM25 advantage)
    "The BM25 algorithm uses term frequency saturation (k1=1.5) and length normalization (b=0.75) to rank documents.",
]

print(f"Corpus loaded: {len(corpus)} documents")

Corpus loaded: 10 documents


## Part 2 — Hybrid Retrieval

In [4]:
import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

class HybridRetriever:
    def __init__(self, corpus: list[str], k: int = 60):
        self.corpus = corpus
        self.k = k

        # BM25 index
        tokenized = [doc.lower().split() for doc in corpus]
        self.bm25 = BM25Okapi(tokenized)

        # SBERT index
        self.sbert = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
        doc_vecs = self.sbert.encode(corpus, convert_to_numpy=True)
        self.doc_vecs = doc_vecs / np.linalg.norm(doc_vecs, axis=1, keepdims=True)

    def retrieve(self, query: str, top_k: int = 5) -> list[dict]:
        # BM25 ranks
        bm25_scores = self.bm25.get_scores(query.lower().split())
        bm25_ranked = np.argsort(bm25_scores)[::-1]
        bm25_ranks  = {int(d): r+1 for r, d in enumerate(bm25_ranked)}

        # SBERT ranks
        q_vec = self.sbert.encode([query], convert_to_numpy=True)[0]
        q_vec = q_vec / np.linalg.norm(q_vec)
        sbert_scores = self.doc_vecs @ q_vec
        sbert_ranked = np.argsort(sbert_scores)[::-1]
        sbert_ranks  = {int(d): r+1 for r, d in enumerate(sbert_ranked)}

        # RRF fusion
        rrf = {}
        for d in range(len(self.corpus)):
            rrf[d] = 1/(self.k + bm25_ranks[d]) + 1/(self.k + sbert_ranks[d])

        final = sorted(rrf, key=rrf.get, reverse=True)[:top_k]
        return [
            {
                "doc_id":     d,
                "rrf_score":  rrf[d],
                "bm25_rank":  bm25_ranks[d],
                "sbert_rank": sbert_ranks[d],
                "text":       self.corpus[d]
            }
            for d in final
        ]

# Test it
retriever = HybridRetriever(corpus)
results = retriever.retrieve("how do transformers encode meaning?")
print("\nHybridRetriever Test:")
for r in results:
    print(f"  doc_{r['doc_id']} | RRF={r['rrf_score']:.5f} | BM25_rank={r['bm25_rank']} | SBERT_rank={r['sbert_rank']}")
    print(f"    {r['text']}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


HybridRetriever Test:
  doc_0 | RRF=0.03279 | BM25_rank=1 | SBERT_rank=1
    Transformers use self-attention mechanisms to compute weighted sums of value vectors based on query-key similarity.
  doc_7 | RRF=0.03150 | BM25_rank=4 | SBERT_rank=3
    Retrieval Augmented Generation (RAG) combines a retriever with a language model to produce grounded, factual answers.
  doc_6 | RRF=0.03128 | BM25_rank=6 | SBERT_rank=2
    BERT is a bidirectional encoder pre-trained using masked language modelling on large text corpora.
  doc_8 | RRF=0.03126 | BM25_rank=3 | SBERT_rank=5
    Neural networks learn by adjusting weights through backpropagation to minimize a task-specific loss function.
  doc_9 | RRF=0.03041 | BM25_rank=2 | SBERT_rank=10
    The BM25 algorithm uses term frequency saturation (k1=1.5) and length normalization (b=0.75) to rank documents.


## Part 3 — Cross-Encoder Re-Ranker

In [5]:
from sentence_transformers import CrossEncoder

cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def rerank(query: str, candidates: list[dict], top_k: int = 3) -> list[dict]:
    """
    Re-ranks candidates using cross-encoder.
    Uses original user query (not HyDE expanded).
    Returns top_k dicts with added 'ce_score'.
    """
    pairs  = [[query, c["text"]] for c in candidates]
    scores = cross_encoder.predict(pairs)

    for i, c in enumerate(candidates):
        c["ce_score"] = float(scores[i])

    reranked = sorted(candidates, key=lambda x: x["ce_score"], reverse=True)
    return reranked[:top_k]

# Test it
reranked = rerank("how do transformers encode meaning?", results)
print("\nCross-Encoder Re-Ranking Test:")
for r in reranked:
    print(f"  CE score={r['ce_score']:.4f} | {r['text']}")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]


Cross-Encoder Re-Ranking Test:
  CE score=-0.8416 | Transformers use self-attention mechanisms to compute weighted sums of value vectors based on query-key similarity.
  CE score=-8.2875 | BERT is a bidirectional encoder pre-trained using masked language modelling on large text corpora.
  CE score=-11.2599 | Retrieval Augmented Generation (RAG) combines a retriever with a language model to produce grounded, factual answers.


## Part 4 — Query Expansion (HyDE)

In [21]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.0)

hyde_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a technical writer. Generate a single factual paragraph (3-5 sentences) that directly answers the following question. Write it as if it were an excerpt from a textbook."),
    ("human", "{query}")
])
hyde_chain = hyde_prompt | llm | StrOutputParser()

def expand_query_hyde(query: str) -> str:
    """Use HyDE to generate a hypothetical answer and return it as the expanded query."""
    return hyde_chain.invoke({"query": query})

# Test it
expanded = expand_query_hyde("how do transformers encode meaning?")
print("\nHyDE Expansion Test:")
print(f"  Original: 'how do transformers encode meaning?'")
print(f"  Expanded: {expanded[:200]}...")


HyDE Expansion Test:
  Original: 'how do transformers encode meaning?'
  Expanded: Transformers encode meaning by first converting input text into numerical representations called token embeddings, which capture initial semantic properties. Positional encodings are then added to the...


## Part 5 — End-to-End Pipeline

In [22]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

generation_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a knowledgeable AI assistant for a university.
Answer the student's question using ONLY the provided context.
If the answer is not in the context, say 'I don't have enough information.'
Be concise and precise.

Context:
{context}"""),
    ("human", "{question}")
])

def advanced_rag(user_query: str) -> str:
    print(f"\nQuery: '{user_query}'")
    print("=" * 65)

    # Step 1: HyDE Query Expansion
    expanded = expand_query_hyde(user_query)
    print(f"[1] HyDE Expansion:\n    {expanded[:150]}...")

    # Step 2: Hybrid Retrieval on expanded query
    candidates = retriever.retrieve(expanded, top_k=5)
    print(f"\n[2] Hybrid Retrieval Top 5:")
    for c in candidates:
        print(f"    doc_{c['doc_id']} | RRF={c['rrf_score']:.5f} | BM25={c['bm25_rank']} | SBERT={c['sbert_rank']}")
        print(f"      {c['text']}")

    # Step 3: Cross-Encoder Re-Ranking (using ORIGINAL query)
    top_docs = rerank(user_query, candidates, top_k=3)
    print(f"\n[3] After Re-Ranking Top 3:")
    for d in top_docs:
        print(f"    CE={d['ce_score']:.4f} | {d['text']}")

    # Step 4: LLM Generation
    context = "\n\n".join(f"[Doc {i+1}] {d['text']}" for i, d in enumerate(top_docs))
    chain   = generation_prompt | llm | StrOutputParser()
    answer  = chain.invoke({"context": context, "question": user_query})

    print(f"\n[4] Final Answer:\n    {answer}")
    return answer

In [23]:
advanced_rag("how does attention work in transformers?")


Query: 'how does attention work in transformers?'
[1] HyDE Expansion:
    In Transformer models, attention, specifically self-attention, allows the model to weigh the importance of different input elements when processing ea...

[2] Hybrid Retrieval Top 5:
    doc_0 | RRF=0.03252 | BM25=2 | SBERT=1
      Transformers use self-attention mechanisms to compute weighted sums of value vectors based on query-key similarity.
    doc_1 | RRF=0.03252 | BM25=1 | SBERT=2
      The attention mechanism allows each token to attend to every other token in the sequence, capturing long-range dependencies.
    doc_2 | RRF=0.03175 | BM25=3 | SBERT=3
      Multi-head attention runs several attention operations in parallel, each learning different relationship patterns.
    doc_8 | RRF=0.03055 | BM25=7 | SBERT=4
      Neural networks learn by adjusting weights through backpropagation to minimize a task-specific loss function.
    doc_6 | RRF=0.03054 | BM25=6 | SBERT=5
      BERT is a bidirectional encoder

'In transformers, attention works by using self-attention mechanisms to compute weighted sums of value vectors based on query-key similarity. This mechanism allows each token to attend to every other token in the sequence.'

## Part 6 — Comparison Experiment

In [12]:
def naive_rag(user_query: str) -> dict:
    """
    Naïve RAG: Dense-only SBERT cosine retrieval, no expansion, no re-ranking.
    Returns top doc info.
    """
    sbert = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
    doc_vecs = sbert.encode(corpus, convert_to_numpy=True)
    doc_vecs = doc_vecs / np.linalg.norm(doc_vecs, axis=1, keepdims=True)

    q_vec = sbert.encode([user_query], convert_to_numpy=True)[0]
    q_vec = q_vec / np.linalg.norm(q_vec)

    scores = doc_vecs @ q_vec
    top_idx = int(np.argmax(scores))
    return {"top_doc": corpus[top_idx], "score": float(scores[top_idx])}

In [13]:
test_queries = [
    "how do transformers encode meaning?",
    "optimization techniques for training",
    "what is BM25 term frequency saturation?",  # own query — tests BM25 jargon doc
]

comparison = []
for q in test_queries:
    print(f"\n{'='*65}")
    print(f"QUERY: {q}")

    naive  = naive_rag(q)
    adv_answer = advanced_rag(q)
    adv_top = rerank(q, retriever.retrieve(expand_query_hyde(q), top_k=5), top_k=1)[0]["text"]

    different = "YES" if naive["top_doc"] != adv_top else "NO"
    comparison.append({
        "query":        q,
        "naive_top":    naive["top_doc"],
        "advanced_top": adv_top,
        "different":    different
    })


QUERY: how do transformers encode meaning?


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Query: 'how do transformers encode meaning?'
[1] HyDE Expansion:
    Transformers encode meaning by first converting input text into numerical tokens, which are then mapped to dense vector embeddings that capture initia...

[2] Hybrid Retrieval Top 5:
    doc_1 | RRF=0.03279 | BM25=1 | SBERT=1
      The attention mechanism allows each token to attend to every other token in the sequence, capturing long-range dependencies.
    doc_0 | RRF=0.03226 | BM25=2 | SBERT=2
      Transformers use self-attention mechanisms to compute weighted sums of value vectors based on query-key similarity.
    doc_8 | RRF=0.03078 | BM25=4 | SBERT=6
      Neural networks learn by adjusting weights through backpropagation to minimize a task-specific loss function.
    doc_6 | RRF=0.03055 | BM25=7 | SBERT=4
      BERT is a bidirectional encoder pre-trained using masked language modelling on large text corpora.
    doc_7 | RRF=0.03054 | BM25=6 | SBERT=5
      Retrieval Augmented Generation (RAG) combines a retr

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Query: 'optimization techniques for training'
[1] HyDE Expansion:
    Optimization techniques are fundamental to efficiently training machine learning models, aiming to minimize the loss function and improve convergence ...

[2] Hybrid Retrieval Top 5:
    doc_3 | RRF=0.03227 | BM25=1 | SBERT=3
      Gradient descent updates model weights by computing the gradient of the loss and stepping in the negative direction.
    doc_8 | RRF=0.03227 | BM25=3 | SBERT=1
      Neural networks learn by adjusting weights through backpropagation to minimize a task-specific loss function.
    doc_5 | RRF=0.03226 | BM25=2 | SBERT=2
      The Adam optimizer combines momentum and adaptive learning rates, making it the default choice for training most LLMs.
    doc_4 | RRF=0.03101 | BM25=5 | SBERT=4
      Backpropagation computes gradients layer by layer using the chain rule, enabling efficient training of deep networks.
    doc_9 | RRF=0.03055 | BM25=4 | SBERT=7
      The BM25 algorithm uses term frequenc

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Query: 'what is BM25 term frequency saturation?'
[1] HyDE Expansion:
    BM25 term frequency saturation refers to a non-linear mechanism within the BM25 ranking function designed to limit the impact of very high term freque...

[2] Hybrid Retrieval Top 5:
    doc_9 | RRF=0.03279 | BM25=1 | SBERT=1
      The BM25 algorithm uses term frequency saturation (k1=1.5) and length normalization (b=0.75) to rank documents.
    doc_0 | RRF=0.03175 | BM25=4 | SBERT=2
      Transformers use self-attention mechanisms to compute weighted sums of value vectors based on query-key similarity.
    doc_7 | RRF=0.03175 | BM25=2 | SBERT=4
      Retrieval Augmented Generation (RAG) combines a retriever with a language model to produce grounded, factual answers.
    doc_1 | RRF=0.03126 | BM25=5 | SBERT=3
      The attention mechanism allows each token to attend to every other token in the sequence, capturing long-range dependencies.
    doc_3 | RRF=0.03037 | BM25=3 | SBERT=9
      Gradient descent updates mo

In [15]:
# ============================================================
# BONUS 1 — Weighted RRF
# ============================================================

def weighted_rrf_retrieve(query: str, alpha: float, top_k: int = 5) -> list[dict]:
    """
    Weighted RRF: gives alpha weight to BM25, (1-alpha) to SBERT.
    alpha=0.7 → BM25 dominant (good for keyword queries)
    alpha=0.3 → SBERT dominant (good for semantic queries)
    """
    # BM25 ranks
    bm25_scores = retriever.bm25.get_scores(query.lower().split())
    bm25_ranked = np.argsort(bm25_scores)[::-1]
    bm25_ranks  = {int(d): r+1 for r, d in enumerate(bm25_ranked)}

    # SBERT ranks
    q_vec = retriever.sbert.encode([query], convert_to_numpy=True)[0]
    q_vec = q_vec / np.linalg.norm(q_vec)
    sbert_scores = retriever.doc_vecs @ q_vec
    sbert_ranked = np.argsort(sbert_scores)[::-1]
    sbert_ranks  = {int(d): r+1 for r, d in enumerate(sbert_ranked)}

    # Weighted RRF
    k = retriever.k
    rrf = {}
    for d in range(len(corpus)):
        rrf[d] = alpha * (1/(k + bm25_ranks[d])) + (1 - alpha) * (1/(k + sbert_ranks[d]))

    final = sorted(rrf, key=rrf.get, reverse=True)[:top_k]
    return [{"doc_id": d, "wrrf_score": rrf[d], "text": corpus[d]} for d in final]


# Test with keyword-heavy query and semantic query
print("=" * 65)
print("BONUS 1 — Weighted RRF Experiment")
print("=" * 65)

keyword_query  = "BM25 term frequency saturation"
semantic_query = "how do neural networks learn from data?"

for alpha in [0.3, 0.5, 0.7]:
    print(f"\nalpha={alpha} (BM25 weight={alpha}, SBERT weight={1-alpha})")
    print(f"  Keyword query top doc:")
    r = weighted_rrf_retrieve(keyword_query, alpha=alpha, top_k=1)
    print(f"    {r[0]['text']}")
    print(f"  Semantic query top doc:")
    r = weighted_rrf_retrieve(semantic_query, alpha=alpha, top_k=1)
    print(f"    {r[0]['text']}")

print("\nObservation: alpha=0.7 (BM25-heavy) helps keyword queries.")
print("Observation: alpha=0.3 (SBERT-heavy) helps semantic queries.")

BONUS 1 — Weighted RRF Experiment

alpha=0.3 (BM25 weight=0.3, SBERT weight=0.7)
  Keyword query top doc:
    The BM25 algorithm uses term frequency saturation (k1=1.5) and length normalization (b=0.75) to rank documents.
  Semantic query top doc:
    Neural networks learn by adjusting weights through backpropagation to minimize a task-specific loss function.

alpha=0.5 (BM25 weight=0.5, SBERT weight=0.5)
  Keyword query top doc:
    The BM25 algorithm uses term frequency saturation (k1=1.5) and length normalization (b=0.75) to rank documents.
  Semantic query top doc:
    Neural networks learn by adjusting weights through backpropagation to minimize a task-specific loss function.

alpha=0.7 (BM25 weight=0.7, SBERT weight=0.30000000000000004)
  Keyword query top doc:
    The BM25 algorithm uses term frequency saturation (k1=1.5) and length normalization (b=0.75) to rank documents.
  Semantic query top doc:
    Neural networks learn by adjusting weights through backpropagation to minimi

In [16]:
# ============================================================
# BONUS 2 — Chunk Size Study
# ============================================================

# A long document (>500 words) to chunk
long_doc = """
Transformers are a type of deep learning model that have revolutionized natural language processing.
They were introduced in the paper Attention is All You Need by Vaswani et al. in 2017.
Unlike recurrent neural networks, transformers process entire sequences in parallel using self-attention.
Self-attention allows each token in the input to attend to every other token, capturing long-range dependencies.
The attention score between two tokens is computed as the dot product of their query and key vectors.
These scores are scaled and passed through a softmax to produce attention weights.
The weighted sum of value vectors gives the final attention output for each token.
Transformers stack multiple such attention layers, each learning different patterns.
Multi-head attention runs several attention operations in parallel with different learned projections.
The outputs of all heads are concatenated and projected back to the original dimension.
Positional encodings are added to token embeddings to give the model information about token order.
BERT uses a bidirectional transformer encoder pre-trained on masked language modelling.
GPT uses a unidirectional transformer decoder pre-trained on next token prediction.
Fine-tuning adapts pre-trained transformer models to specific downstream tasks.
LoRA is a popular parameter-efficient fine-tuning method that adds low-rank adapter matrices.
Quantization reduces model size by storing weights in lower bit formats like INT8 or INT4.
The BM25 algorithm is a classical sparse retrieval method based on term frequency and inverse document frequency.
RAG combines a retriever with a language model to produce grounded, factual answers.
Vector databases like FAISS store dense embeddings for fast approximate nearest neighbour search.
Hybrid retrieval combines sparse BM25 and dense SBERT with Reciprocal Rank Fusion for best results.
Cross-encoders re-rank retrieved candidates by jointly encoding query and document for precise relevance scoring.
HyDE generates a hypothetical answer to a query and uses its embedding for retrieval instead of the raw query.
"""

def chunk_text(text: str, chunk_size_words: int) -> list[str]:
    """Split text into chunks of chunk_size_words words."""
    words  = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size_words):
        chunk = " ".join(words[i:i + chunk_size_words])
        if chunk.strip():
            chunks.append(chunk.strip())
    return chunks

print("=" * 65)
print("BONUS 2 — Chunk Size Study")
print("=" * 65)

test_query = "how does self-attention work in transformers?"

for chunk_size in [50, 100, 200]:
    chunks   = chunk_text(long_doc, chunk_size)
    temp_ret = HybridRetriever(chunks)
    results  = temp_ret.retrieve(test_query, top_k=1)
    print(f"\nChunk size = {chunk_size} words → {len(chunks)} chunks")
    print(f"  Top chunk: {results[0]['text'][:120]}...")
    print(f"  RRF score: {results[0]['rrf_score']:.5f}")

print("\nObservation: Smaller chunks are more precise but may lose context.")
print("Observation: Larger chunks retain more context but may dilute relevance score.")

BONUS 2 — Chunk Size Study


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Chunk size = 50 words → 7 chunks
  Top chunk: Transformers are a type of deep learning model that have revolutionized natural language processing. They were introduce...
  RRF score: 0.03279


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Chunk size = 100 words → 4 chunks
  Top chunk: Transformers are a type of deep learning model that have revolutionized natural language processing. They were introduce...
  RRF score: 0.03279


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Chunk size = 200 words → 2 chunks
  Top chunk: Transformers are a type of deep learning model that have revolutionized natural language processing. They were introduce...
  RRF score: 0.03252

Observation: Smaller chunks are more precise but may lose context.
Observation: Larger chunks retain more context but may dilute relevance score.


In [18]:
# ============================================================
# BONUS 3 — ColBERT as 3rd Retriever in Hybrid RRF
# ============================================================

from sentence_transformers import SentenceTransformer

colbert_encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

def colbert_score(query: str, document: str) -> float:
    """ColBERT MaxSim: each query token finds best matching doc token, sum those max scores."""
    q_tokens = query.lower().split()
    d_tokens = document.lower().split()

    q_vecs = colbert_encoder.encode(q_tokens, convert_to_numpy=True)
    d_vecs = colbert_encoder.encode(d_tokens, convert_to_numpy=True)

    q_vecs = q_vecs / np.linalg.norm(q_vecs, axis=1, keepdims=True)
    d_vecs = d_vecs / np.linalg.norm(d_vecs, axis=1, keepdims=True)

    sim_matrix = q_vecs @ d_vecs.T
    max_sims   = sim_matrix.max(axis=1)
    return float(max_sims.sum())


def hybrid_retrieve_3way(query: str, top_k: int = 5) -> list[dict]:
    """
    3-way Hybrid: BM25 + SBERT + ColBERT fused with RRF.
    """
    k = retriever.k

    # BM25 ranks
    bm25_scores = retriever.bm25.get_scores(query.lower().split())
    bm25_ranked = np.argsort(bm25_scores)[::-1]
    bm25_ranks  = {int(d): r+1 for r, d in enumerate(bm25_ranked)}

    # SBERT ranks
    q_vec = retriever.sbert.encode([query], convert_to_numpy=True)[0]
    q_vec = q_vec / np.linalg.norm(q_vec)
    sbert_scores = retriever.doc_vecs @ q_vec
    sbert_ranked = np.argsort(sbert_scores)[::-1]
    sbert_ranks  = {int(d): r+1 for r, d in enumerate(sbert_ranked)}

    # ColBERT ranks
    colbert_scores = [colbert_score(query, doc) for doc in corpus]
    colbert_ranked = np.argsort(colbert_scores)[::-1]
    colbert_ranks  = {int(d): r+1 for r, d in enumerate(colbert_ranked)}

    # 3-way RRF fusion
    rrf = {}
    for d in range(len(corpus)):
        rrf[d] = (
            1/(k + bm25_ranks[d]) +
            1/(k + sbert_ranks[d]) +
            1/(k + colbert_ranks[d])
        )

    final = sorted(rrf, key=rrf.get, reverse=True)[:top_k]
    return [
        {
            "doc_id":        d,
            "rrf_score":     rrf[d],
            "bm25_rank":     bm25_ranks[d],
            "sbert_rank":    sbert_ranks[d],
            "colbert_rank":  colbert_ranks[d],
            "text":          corpus[d]
        }
        for d in final
    ]


# Test it
print("=" * 65)
print("BONUS 3 — ColBERT as 3rd Retriever")
print("=" * 65)

query = "how does attention mechanism work?"
print(f"\nQuery: '{query}'")

print("\n2-way Hybrid (BM25 + SBERT):")
for r in retriever.retrieve(query, top_k=3):
    print(f"  doc_{r['doc_id']} RRF={r['rrf_score']:.5f} | {r['text'][:70]}")

print("\n3-way Hybrid (BM25 + SBERT + ColBERT):")
for r in hybrid_retrieve_3way(query, top_k=3):
    print(f"  doc_{r['doc_id']} RRF={r['rrf_score']:.5f} | BM25={r['bm25_rank']} SBERT={r['sbert_rank']} ColBERT={r['colbert_rank']} | {r['text'][:70]}")

print("Observation: ColBERT adds token-level matching on top of BM25 and SBERT.")
print("Observation: 3-way RRF gives more robust rankings across query types.")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BONUS 3 — ColBERT as 3rd Retriever

Query: 'how does attention mechanism work?'

2-way Hybrid (BM25 + SBERT):
  doc_1 RRF=0.03279 | The attention mechanism allows each token to attend to every other tok
  doc_2 RRF=0.03226 | Multi-head attention runs several attention operations in parallel, ea
  doc_8 RRF=0.03175 | Neural networks learn by adjusting weights through backpropagation to 

3-way Hybrid (BM25 + SBERT + ColBERT):
  doc_1 RRF=0.04918 | BM25=1 SBERT=1 ColBERT=1 | The attention mechanism allows each token to attend to every other tok
  doc_2 RRF=0.04813 | BM25=2 SBERT=2 ColBERT=3 | Multi-head attention runs several attention operations in parallel, ea
  doc_8 RRF=0.04713 | BM25=3 SBERT=3 ColBERT=5 | Neural networks learn by adjusting weights through backpropagation to 
Observation: ColBERT adds token-level matching on top of BM25 and SBERT.
Observation: 3-way RRF gives more robust rankings across query types.


## Comparison Table

| Query | Naïve RAG Top Doc | Advanced RAG Top Doc | Are they different? |
|---|---|---|---|
| how do transformers encode meaning? | Transformers use self-attention mechanisms to compute weight... | Transformers use self-attention mechanisms to compute weight... | NO |
| optimization techniques for training | Neural networks learn by adjusting weights through backpropa... | The Adam optimizer combines momentum and adaptive learning r... | YES |
| what is BM25 term frequency saturation? | The BM25 algorithm uses term frequency saturation (k1=1.5) a... | The BM25 algorithm uses term frequency saturation (k1=1.5) a... | NO |

## Assignment Summary & Observations
- Hybrid retrieval improved keyword matching for jargon-heavy questions.
- HyDE improved retrieval for vague student questions.
- Cross-encoder improved final ranking quality by removing semantically similar but less relevant docs.
- BM25 contributed strongly for the BM25-specific query where dense retrieval alone was weaker.